# NeuroGraph-X · Week 1 · Brain-shaped synthetic test
Owner: Laiba Afridi. **Synthetic data, not ABIDE.**

Runs the training loop at real brain size (200 nodes, 200 features) and rehearses the evaluation protocol:
frozen set drawn first, stratified train/val split, checkpoint chosen on validation, one measurement on the frozen set.

- `--effect 0.5`: planted signal, the model should learn it.
- `--effect 0.0`: no signal (null control), frozen-set accuracy must be near 50%.

**Before running:** Accelerator = GPU T4, Internet = On. Then *Run All*.

## 1. Install and check the GPU

In [ ]:
!pip install -q torch_geometric pytest

In [ ]:
import torch
import torch_geometric
print("PyTorch:", torch.__version__, "| PyG:", torch_geometric.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on GPU T4")

## 2. Week 1 files (same as in the repository)

In [ ]:
%%writefile run_logger.py
"""Minimal run logger (seed of L5).

Every training run appends ONE JSON line to runs/run_log.jsonl containing its
configuration and result. Later this maps directly onto Alishba's run-log
table in the database schema.
"""
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

LOG_PATH = Path("runs/run_log.jsonl")


def new_run_id() -> str:
    return uuid.uuid4().hex[:8]


def log_run(record: dict, path: Path = LOG_PATH) -> str:
    """Append a run record. `record` should contain run_id, dataset, atlas,
    seed, hyperparameters, metrics and checkpoint_path."""
    path.parent.mkdir(parents=True, exist_ok=True)
    record = {
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        **record,
    }
    with open(path, "a") as f:
        f.write(json.dumps(record) + "\n")
    return record["run_id"]


def read_runs(path: Path = LOG_PATH) -> list[dict]:
    if not path.exists():
        return []
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

In [ ]:
%%writefile train_toy.py
"""Week 1 toy training loop. Proves training, logging, checkpointing and
reproducibility work BEFORE real ABIDE data arrives.

Usage:
    python train_toy.py                       # MUTAG, defaults
    python train_toy.py --dataset PROTEINS --epochs 150 --seed 7
"""
import argparse
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from run_logger import log_run, new_run_id
from torch.nn import Dropout, Linear, ModuleList, ReLU, Sequential
from torch_geometric.datasets import FakeDataset, TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool


# ----------------------------------------------------------------- utilities
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# --------------------------------------------------------------------- model
class GCN(torch.nn.Module):
    """N x GCNConv -> mean-pool readout -> MLP head."""

    def __init__(self, in_dim, hidden, num_classes, num_layers=3, dropout=0.5):
        super().__init__()
        dims = [in_dim] + [hidden] * num_layers
        self.convs = ModuleList(
            [GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)]
        )
        self.head = Sequential(
            Linear(hidden, hidden), ReLU(), Dropout(dropout), Linear(hidden, num_classes)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)  # graph-level readout
        return self.head(x)  # logits


# ---------------------------------------------------------------- train/eval
def train_epoch(model, loader, optimizer, device):
    model.train()
    total = 0.0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(data.x, data.edge_index, data.batch), data.y)
        loss.backward()
        optimizer.step()
        total += loss.item() * data.num_graphs
    return total / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, device):
    """Returns (accuracy, logits) with logits in loader order."""
    model.eval()
    correct, all_logits = 0, []
    for data in loader:
        data = data.to(device)
        logits = model(data.x, data.edge_index, data.batch)
        correct += (logits.argmax(dim=1) == data.y).sum().item()
        all_logits.append(logits.cpu())
    return correct / len(loader.dataset), torch.cat(all_logits)


# ---------------------------------------------------------------------- main
def main():
    p = argparse.ArgumentParser()
    p.add_argument("--dataset", default="MUTAG")
    p.add_argument("--epochs", type=int, default=100)
    p.add_argument("--hidden", type=int, default=64)
    p.add_argument("--layers", type=int, default=3)
    p.add_argument("--dropout", type=float, default=0.5)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--batch_size", type=int, default=32)
    p.add_argument("--seed", type=int, default=42)
    args = p.parse_args()

    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    if args.dataset.upper() == "FAKE":
        # Offline synthetic graphs, no download. Labels are random, so accuracy
        # will hover near 50%. Use only to prove the plumbing works.
        dataset = FakeDataset(num_graphs=200, avg_num_nodes=30, avg_degree=4,
                              num_channels=7, num_classes=2)
    else:
        try:
            dataset = TUDataset(root="data/TUDataset", name=args.dataset)
        except Exception as e:  # noqa: BLE001
            raise SystemExit(
                f"Could not download {args.dataset}: {e}\n"
                "Turn Internet ON in Kaggle (Session options > Internet), "
                "or run with --dataset FAKE for an offline test."
            )
    print(f"{args.dataset}: {len(dataset)} graphs, "
          f"{dataset.num_node_features} node feats, {dataset.num_classes} classes")

    # Fixed, seeded 70/15/15 split (train / val / test)
    g = torch.Generator().manual_seed(args.seed)
    perm = torch.randperm(len(dataset), generator=g)
    n_train, n_val = int(0.7 * len(dataset)), int(0.15 * len(dataset))
    train_ds = dataset[perm[:n_train]]
    val_ds = dataset[perm[n_train:n_train + n_val]]
    test_ds = dataset[perm[n_train + n_val:]]

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size)
    test_loader = DataLoader(test_ds, batch_size=args.batch_size)

    model = GCN(dataset.num_node_features, args.hidden, dataset.num_classes,
                args.layers, args.dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

    run_id = new_run_id()
    ckpt_path = Path("checkpoints") / f"{args.dataset}_{run_id}.pt"
    ckpt_path.parent.mkdir(exist_ok=True)

    best_val, best_epoch = -1.0, 0
    for epoch in range(1, args.epochs + 1):
        loss = train_epoch(model, train_loader, optimizer, device)
        val_acc, _ = evaluate(model, val_loader, device)
        if val_acc > best_val:  # select checkpoint by VALIDATION accuracy
            best_val, best_epoch = val_acc, epoch
            torch.save({
                "state_dict": model.state_dict(),
                "in_dim": dataset.num_node_features,
                "num_classes": dataset.num_classes,
                "hidden": args.hidden,
                "layers": args.layers,
                "dropout": args.dropout,
            }, ckpt_path)
        if epoch % 10 == 0 or epoch == 1:
            print(f"epoch {epoch:3d} | loss {loss:.4f} | val_acc {val_acc:.3f}")

    # ---- reload best checkpoint, evaluate on held-out test split
    ckpt = torch.load(ckpt_path, map_location=device)
    best_model = GCN(ckpt["in_dim"], ckpt["hidden"], ckpt["num_classes"],
                     ckpt["layers"], ckpt["dropout"]).to(device)
    best_model.load_state_dict(ckpt["state_dict"])
    test_acc, logits_a = evaluate(best_model, test_loader, device)

    # ---- reproducibility test: reload a second time, logits must be identical
    best_model2 = GCN(ckpt["in_dim"], ckpt["hidden"], ckpt["num_classes"],
                      ckpt["layers"], ckpt["dropout"]).to(device)
    best_model2.load_state_dict(torch.load(ckpt_path, map_location=device)["state_dict"])
    _, logits_b = evaluate(best_model2, test_loader, device)
    identical = torch.allclose(logits_a, logits_b, atol=1e-6)

    print(f"\nBest val acc {best_val:.3f} at epoch {best_epoch}")
    print(f"Test acc     {test_acc:.3f}")
    print(f"Reload reproduces identical predictions: {identical}")

    log_run({
        "run_id": run_id,
        "dataset": args.dataset,
        "atlas": None,  # filled in for ABIDE runs (e.g. "cc200")
        "seed": args.seed,
        "hyperparameters": {
            "hidden": args.hidden, "layers": args.layers, "dropout": args.dropout,
            "lr": args.lr, "batch_size": args.batch_size, "epochs": args.epochs,
        },
        "metrics": {"best_val_acc": best_val, "best_epoch": best_epoch,
                    "test_acc": test_acc},
        "checkpoint_path": str(ckpt_path),
        "reload_identical": identical,
    })
    print("Run logged. run_id =", run_id)


if __name__ == "__main__":
    main()

## 3. Synthetic brain generator and its tests

In [ ]:
%%writefile synthetic_brain.py
"""Brain-shaped synthetic graphs for NeuroGraph-X pipeline testing.
NOT real data. Used to test shapes, thresholding and the training loop
before ABIDE matrices arrive. matrix_to_graph() is reused on real data."""
import numpy as np
import torch
from torch_geometric.data import Data


def assign_networks(n_rois, n_networks, seed=0):
    """Fixed ROI-to-network assignment, every network non-empty."""
    rng = np.random.default_rng(seed)
    return rng.permutation(np.arange(n_rois) % n_networks)


def simulate_timeseries(rng, network_of_roi, n_timepoints, label, effect,
                        base_coupling=0.8, noise_sd=1.0):
    """Label 1 has weaker coupling between networks 0 and 1 (planted signal)."""
    n_networks = int(network_of_roi.max()) + 1
    latent = rng.standard_normal((n_timepoints, n_networks))
    coupling = base_coupling * (1.0 - effect * label) + rng.normal(0.0, 0.1)
    shared = rng.standard_normal(n_timepoints)
    latent[:, 0] += coupling * shared
    latent[:, 1] += coupling * shared
    noise = noise_sd * rng.standard_normal((n_timepoints, len(network_of_roi)))
    return latent[:, network_of_roi] + noise


def correlation_to_fisher_z(ts):
    """Pearson correlation between ROI columns, Fisher z, diagonal set to 0."""
    r = np.corrcoef(ts, rowvar=False)
    r = np.clip(r, -0.999999, 0.999999)
    z = np.arctanh(r)
    np.fill_diagonal(z, 0.0)
    return z.astype(np.float32)


def matrix_to_graph(z, label, density=0.10, subject_id=None, site=None,
                    atlas="synthetic"):
    """Unthresholded N x N Fisher-z matrix -> PyG graph.
    Node features: correlation rows. Edges: top `density` fraction of |z|
    (proportional threshold), undirected, weight = |z|."""
    z = np.asarray(z, dtype=np.float32)
    n = z.shape[0]
    if z.ndim != 2 or z.shape != (n, n):
        raise ValueError(f"expected square matrix, got {z.shape}")
    if not np.isfinite(z).all():
        raise ValueError(f"{subject_id}: matrix contains NaN or inf")
    if not np.allclose(z, z.T, atol=1e-5):
        raise ValueError(f"{subject_id}: matrix is not symmetric")
    iu, ju = np.triu_indices(n, k=1)
    strength = np.abs(z[iu, ju])
    k = max(1, round(density * strength.size))
    keep = np.argpartition(strength, -k)[-k:]
    src = np.concatenate([iu[keep], ju[keep]])
    dst = np.concatenate([ju[keep], iu[keep]])
    return Data(
        x=torch.from_numpy(z.copy()),
        edge_index=torch.tensor(np.stack([src, dst]), dtype=torch.long),
        edge_weight=torch.tensor(np.concatenate([strength[keep], strength[keep]]),
                                 dtype=torch.float32),
        y=torch.tensor([int(label)], dtype=torch.long),
        num_nodes=n, subject_id=subject_id, site=site, atlas=atlas,
    )


def make_dataset(n_subjects=200, n_rois=200, n_timepoints=150, n_networks=7,
                 effect=0.5, density=0.10, n_sites=4, seed=0):
    """Balanced synthetic cohort. effect=0 means no signal (null control)."""
    rng = np.random.default_rng(seed)
    network_of_roi = assign_networks(n_rois, n_networks, seed)
    labels = np.array([i % 2 for i in range(n_subjects)])
    rng.shuffle(labels)
    sites = rng.integers(0, n_sites, size=n_subjects)
    graphs = []
    for i in range(n_subjects):
        site_noise = 1.0 + 0.3 * sites[i] / max(1, n_sites - 1)
        ts = simulate_timeseries(rng, network_of_roi, n_timepoints,
                                 int(labels[i]), effect, noise_sd=site_noise)
        graphs.append(matrix_to_graph(
            correlation_to_fisher_z(ts), int(labels[i]), density,
            subject_id=f"SYN_{i:04d}", site=f"SITE{sites[i]}",
            atlas=f"synthetic{n_rois}"))
    return graphs

In [ ]:
!mkdir -p tests

In [ ]:
%%writefile tests/test_synthetic_graphs.py
import numpy as np
import pytest
import torch
from synthetic_brain import make_dataset, matrix_to_graph


def test_shapes_and_validity():
    for g in make_dataset(n_subjects=10, n_rois=200, seed=1):
        assert g.x.shape == (200, 200)
        assert torch.isfinite(g.x).all()
        assert (g.edge_index[0] != g.edge_index[1]).all()  # no self-loops
        assert (g.edge_weight >= 0).all()
        assert g.edge_index.shape[1] == 2 * round(0.10 * 200 * 199 / 2)


def test_edges_are_symmetric():
    g = make_dataset(n_subjects=1, seed=2)[0]
    pairs = set(map(tuple, g.edge_index.t().tolist()))
    assert all((j, i) in pairs for (i, j) in pairs)


def test_other_node_count_aal_sized():
    g = make_dataset(n_subjects=2, n_rois=116, seed=3)[0]
    assert g.num_nodes == 116 and g.x.shape == (116, 116)


def test_rejects_nan_matrix():
    z = np.zeros((5, 5), dtype=np.float32)
    z[0, 1] = z[1, 0] = np.nan
    with pytest.raises(ValueError):
        matrix_to_graph(z, label=0)


def test_same_seed_same_data():
    a = make_dataset(n_subjects=3, seed=4)
    b = make_dataset(n_subjects=3, seed=4)
    assert all(torch.equal(g1.x, g2.x) for g1, g2 in zip(a, b))

In [ ]:
!python -m pytest -q tests/test_synthetic_graphs.py

Expected: `5 passed`.

In [ ]:
import numpy as np
from synthetic_brain import make_dataset
graphs = make_dataset(n_subjects=200, effect=0.5, seed=0)
print(graphs[0])
print("labels:", np.bincount([int(g.y) for g in graphs]))
print("feature memory: %.1f MB" % (sum(g.x.numel() * 4 for g in graphs) / 1e6))

Expected: `x=[200, 200]`, `edge_index=[2, 3980]`, labels `[100 100]`, about 32 MB.

## 4. Training on synthetic brains

In [ ]:
%%writefile train_synthetic.py
"""Week 1 brain-shaped synthetic test (NOT real data).
Rehearses the ABIDE protocol on synthetic brains: a frozen set drawn first and
never used for training or checkpoint selection, a stratified train/val split
of the remaining pool, checkpoint chosen by validation accuracy, then ONE
measurement on the frozen set. Run with --effect 0.0 as the null control.

Usage:
    python train_synthetic.py --effect 0.5
    python train_synthetic.py --effect 0.0
"""
import argparse
from pathlib import Path

import numpy as np
import torch
from run_logger import log_run, new_run_id
from sklearn.model_selection import train_test_split
from synthetic_brain import make_dataset
from torch_geometric.loader import DataLoader
from train_toy import GCN, evaluate, set_seed, train_epoch


def split_indices(graphs, frozen_n, val_frac, seed):
    """Frozen set first, then train/val from the remaining pool.
    Both splits stratified by site and label together."""
    ids = np.array([g.subject_id for g in graphs])
    strata = np.array([f"{g.site}_{int(g.y)}" for g in graphs])
    pool_idx, frozen_idx = train_test_split(
        np.arange(len(graphs)), test_size=frozen_n,
        stratify=strata, random_state=seed)
    train_idx, val_idx = train_test_split(
        pool_idx, test_size=val_frac,
        stratify=strata[pool_idx], random_state=seed + 1)
    # Leakage assertions: no subject in more than one split
    assert not set(ids[frozen_idx]) & set(ids[pool_idx]), "frozen leak"
    assert not set(ids[train_idx]) & set(ids[val_idx]), "train/val overlap"
    return train_idx, val_idx, frozen_idx


def main():
    p = argparse.ArgumentParser()
    # data
    p.add_argument("--effect", type=float, default=0.5)
    p.add_argument("--n_subjects", type=int, default=400)
    p.add_argument("--n_rois", type=int, default=200)
    p.add_argument("--density", type=float, default=0.10)
    p.add_argument("--frozen_n", type=int, default=100)
    p.add_argument("--val_frac", type=float, default=0.2)
    p.add_argument("--data_seed", type=int, default=0)
    # model (same defaults as train_toy.py)
    p.add_argument("--epochs", type=int, default=100)
    p.add_argument("--hidden", type=int, default=64)
    p.add_argument("--layers", type=int, default=3)
    p.add_argument("--dropout", type=float, default=0.5)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--batch_size", type=int, default=32)
    p.add_argument("--seed", type=int, default=42)
    args = p.parse_args()

    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    graphs = make_dataset(n_subjects=args.n_subjects, n_rois=args.n_rois,
                          effect=args.effect, density=args.density,
                          seed=args.data_seed)
    train_idx, val_idx, frozen_idx = split_indices(
        graphs, args.frozen_n, args.val_frac, args.data_seed)
    train_ds = [graphs[i] for i in train_idx]
    val_ds = [graphs[i] for i in val_idx]
    frozen_ds = [graphs[i] for i in frozen_idx]
    for name, ds in [("train", train_ds), ("val", val_ds), ("frozen", frozen_ds)]:
        counts = np.bincount([int(g.y) for g in ds], minlength=2)
        print(f"{name:6s}: {len(ds):3d} subjects | control {counts[0]} | case {counts[1]}")

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size)
    frozen_loader = DataLoader(frozen_ds, batch_size=args.batch_size)

    model = GCN(args.n_rois, args.hidden, 2, args.layers, args.dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

    run_id = new_run_id()
    ckpt_path = Path("checkpoints") / f"synthetic_e{args.effect}_{run_id}.pt"
    ckpt_path.parent.mkdir(exist_ok=True)

    best_val, best_epoch = -1.0, 0
    for epoch in range(1, args.epochs + 1):
        loss = train_epoch(model, train_loader, optimizer, device)
        val_acc, _ = evaluate(model, val_loader, device)
        if val_acc > best_val:  # checkpoint chosen on VALIDATION only
            best_val, best_epoch = val_acc, epoch
            torch.save({
                "state_dict": model.state_dict(),
                "in_dim": args.n_rois, "num_classes": 2,
                "hidden": args.hidden, "layers": args.layers,
                "dropout": args.dropout,
            }, ckpt_path)
        if epoch % 10 == 0 or epoch == 1:
            print(f"epoch {epoch:3d} | loss {loss:.4f} | val_acc {val_acc:.3f}")

    # ---- ONE measurement on the frozen set, with the selected checkpoint
    ckpt = torch.load(ckpt_path, map_location=device)
    best_model = GCN(ckpt["in_dim"], ckpt["hidden"], ckpt["num_classes"],
                     ckpt["layers"], ckpt["dropout"]).to(device)
    best_model.load_state_dict(ckpt["state_dict"])
    frozen_acc, logits_a = evaluate(best_model, frozen_loader, device)
    pred_counts = np.bincount(logits_a.argmax(dim=1).numpy(), minlength=2).tolist()

    # ---- reproducibility: reload again, logits must match
    best_model2 = GCN(ckpt["in_dim"], ckpt["hidden"], ckpt["num_classes"],
                      ckpt["layers"], ckpt["dropout"]).to(device)
    best_model2.load_state_dict(torch.load(ckpt_path, map_location=device)["state_dict"])
    _, logits_b = evaluate(best_model2, frozen_loader, device)
    identical = torch.allclose(logits_a, logits_b, atol=1e-6)

    print(f"\nBest val acc     {best_val:.3f} at epoch {best_epoch}")
    print(f"Frozen-set acc   {frozen_acc:.3f}  (predicted control/case: {pred_counts})")
    print(f"Reload reproduces identical predictions: {identical}")

    log_run({
        "run_id": run_id,
        "dataset": "synthetic_brain",
        "atlas": f"synthetic{args.n_rois}",
        "seed": args.seed,
        "data_config": {
            "effect": args.effect, "n_subjects": args.n_subjects,
            "n_rois": args.n_rois, "density": args.density,
            "frozen_n": args.frozen_n, "val_frac": args.val_frac,
            "data_seed": args.data_seed,
        },
        "hyperparameters": {
            "hidden": args.hidden, "layers": args.layers, "dropout": args.dropout,
            "lr": args.lr, "batch_size": args.batch_size, "epochs": args.epochs,
        },
        "split_sizes": {"train": len(train_ds), "val": len(val_ds),
                        "frozen": len(frozen_ds)},
        "metrics": {"best_val_acc": best_val, "best_epoch": best_epoch,
                    "frozen_eval_acc": frozen_acc,
                    "frozen_pred_counts": pred_counts},
        "checkpoint_path": str(ckpt_path),
        "reload_identical": bool(identical),
    })
    print("Run logged. run_id =", run_id)


if __name__ == "__main__":
    main()

In [ ]:
!python train_synthetic.py --effect 0.5

Expected: splits 240 / 60 / 100, frozen-set accuracy clearly above chance (about 0.85-0.90).

In [ ]:
!python train_synthetic.py --effect 0.0

Expected: frozen-set accuracy about 0.40-0.60 (chance). Best validation accuracy may look a bit higher: picking the best of 100 epochs on 60 subjects flatters it, which is why validation accuracy is never the reported result.

In [ ]:
!tail -n 2 runs/run_log.jsonl

## 5. Evidence
Screenshot the pytest result, both training outputs and the two log lines. Then **Save Version** and download `runs/run_log.jsonl` and this notebook.